# 02613 F25 Exam — Code Proofs
This is the **primary practice exam** (same MCQ-only format as the real exam).
Each cell proves the answer by running the actual computation.


In [1]:
import numpy as np


## Q1 — rusage memory per core (4 cores, 100 GB total)
Answer: **A) 25 GB**


In [2]:
total_GB = 100
n_cores  = 4
per_core_GB = total_GB / n_cores
print(f'rusage[mem={per_core_GB:.0f}GB]')  # 25GB


rusage[mem=25GB]


## Q2 — float16 precision: what does `float16(10000) + float16(1)` print?
Answer: **B) 10000.0** (ULP at 10000 ≈ 10, so 1 is too small to register)


In [3]:
x = np.float16(10000)
y = np.float16(1)
result = x + y
print(f'float16(10000) + float16(1) = {result}')  # 10000.0

# Why: ULP (unit in last place) at 10000 in float16
eps = np.finfo(np.float16).eps
print(f'float16 machine epsilon (relative): {eps}')  # ~0.001
print(f'ULP at 10000 ≈ {10000 * eps:.1f}')           # ≈ 10
print(f'Next representable value above 10000: {np.nextafter(np.float16(10000), np.float16(np.inf))}')


float16(10000) + float16(1) = 10000.0
float16 machine epsilon (relative): 0.0009765625
ULP at 10000 ≈ 9.8
Next representable value above 10000: 10008.0


## Q3 — Can set intersection be used in parallel reduction?
Answer: **A) Yes** (associative AND commutative)


In [4]:
A = {1, 2, 3, 4}
B = {2, 3, 4, 5}
C = {3, 4, 5, 6}

# Associative: (A ∩ B) ∩ C == A ∩ (B ∩ C)
left  = (A & B) & C
right = A & (B & C)
print(f'(A∩B)∩C = {sorted(left)}')
print(f'A∩(B∩C) = {sorted(right)}')
print(f'Associative: {left == right}')  # True

# Commutative: A ∩ B == B ∩ A
print(f'Commutative: {(A & B) == (B & A)}')  # True
print('→ Valid for parallel reduction')


(A∩B)∩C = [3, 4]
A∩(B∩C) = [3, 4]
Associative: True
Commutative: True
→ Valid for parallel reduction


## Q4 — `a.reshape(-1)[8]` for a 3×5 row-wise array
Answer: **A) 67**


In [5]:
a = np.array([[1, 5, 43, 51, 32],
              [73, 2, 4, 67, 37],
              [9, 3, 54, 8, 22]])

flat = a.reshape(-1)
print(f'Flattened (row-major): {flat}')
print(f'Index 8: {flat[8]}')  # 67


Flattened (row-major): [ 1  5 43 51 32 73  2  4 67 37  9  3 54  8 22]
Index 8: 67


## Q5 — Broadcasting: images (N,H,W,3) - mean_pixels (N,3)
Answer: **A) `images - mean_pixels[:, None, None]`**


In [6]:
N, H, W = 8, 32, 32
images      = np.zeros((N, H, W, 3))
mean_pixels = np.ones((N, 3))

# Correct
result = images - mean_pixels[:, None, None]
print(f'Correct shape: {result.shape}')  # (8, 32, 32, 3)

# Wrong attempt
try:
    bad = images - mean_pixels
    print(f'Wrong attempt shape: {bad.shape}')  # may error
except Exception as e:
    print(f'Wrong attempt: ERROR — {e}')

# Show why the None insertion works
print(f'mean_pixels[:,None,None] shape: {mean_pixels[:,None,None].shape}')  # (8,1,1,3)


Correct shape: (8, 32, 32, 3)
Wrong attempt: ERROR — operands could not be broadcast together with shapes (8,32,32,3) (8,3) 
mean_pixels[:,None,None] shape: (8, 1, 1, 3)


## Q8 — Amdahl's law: S(3)=2.5 → find F and S_max
Answer: **C) ~10x**


In [7]:
p = 3
S_measured = 2.5

F = p * (1 - 1/S_measured) / (p - 1)
S_max = 1 / (1 - F)

print(f'F     = {F}')         # 0.9
print(f'S_max = {S_max:.1f}') # 10.0

# Verify
S3_check = 1 / ((1 - F) + F/p)
print(f'S(3) check = {S3_check}')  # should be 2.5


F     = 0.8999999999999999
S_max = 10.0
S(3) check = 2.4999999999999996


## Q9 — `time` command: real/user/sys with 2 cores
Single: real=12s, user=12s. On 2 cores: real≈6s, user≈12s (stays same)


In [8]:
# Demonstrate with actual parallel work
import subprocess, sys

print('Rule: real decreases with parallelism; user stays same (summed over cores)')
print()
print('Single-threaded:    real=12s, user=12s, sys=0.034s')
print('2-core parallel:    real≈6s,  user≈12s, sys=0.034s  ← only real changes')
print()
print('user = CPU time × n_cores ≈ 6s × 2 = 12s (stays constant or increases)')


Rule: real decreases with parallelism; user stays same (summed over cores)

Single-threaded:    real=12s, user=12s, sys=0.034s
2-core parallel:    real≈6s,  user≈12s, sys=0.034s  ← only real changes

user = CPU time × n_cores ≈ 6s × 2 = 12s (stays constant or increases)


## Q10 — Static vs dynamic scheduling (stddev of kernels)
kernel1 stddev=40ms (high) → dynamic. kernel2 stddev=0.05ms (low) → static


In [9]:
import numpy as np

kernel1_times = np.random.normal(200, 40, 100)   # high stddev
kernel2_times = np.random.normal(200, 0.05, 100) # low stddev

print(f'kernel1 stddev: {kernel1_times.std():.1f}ms → dynamic scheduling (high variance)')
print(f'kernel2 stddev: {kernel2_times.std():.3f}ms → static scheduling (uniform tasks)')

# Why: with 4 GPUs and static scheduling, one GPU gets 'slow' tasks by chance
# and becomes the bottleneck. Dynamic hands out one task at a time.


kernel1 stddev: 39.3ms → dynamic scheduling (high variance)
kernel2 stddev: 0.050ms → static scheduling (uniform tasks)


## Q11 — Line profiler: project to normal workload (10000 entries)
Answer: **A) ~14.7 seconds**


In [10]:
# Profiled with 1000 entries:
prep_conds_time_s = 2.005   # fixed cost (doesn't scale)
loop_time_us = 740 + 1_266_748 + 1_685  # microseconds for 1000 iterations
loop_time_s = loop_time_us / 1e6

production_entries = 10_000
profile_entries    = 1_000

loop_projected = loop_time_s * (production_entries / profile_entries)
total_projected = prep_conds_time_s + loop_projected

print(f'prep_conds (fixed):  {prep_conds_time_s:.3f}s')
print(f'loop (1000 entries): {loop_time_s:.3f}s')
print(f'loop (10000 entries):{loop_projected:.3f}s')
print(f'Total projected:     {total_projected:.2f}s')  # ~14.7s


prep_conds (fixed):  2.005s
loop (1000 entries): 1.269s
loop (10000 entries):12.692s
Total projected:     14.70s


## Q13 — Number of 16×16 thread blocks for 200×200 output
Answer: **A) 13×13**


In [11]:
import math

output_size = 200
block_size  = 16

blocks_per_dim = math.ceil(output_size / block_size)
total_blocks   = blocks_per_dim ** 2

print(f'ceil({output_size}/{block_size}) = {blocks_per_dim}')  # 13
print(f'Total blocks: {blocks_per_dim}×{blocks_per_dim} = {total_blocks}')  # 13×13=169
print(f'Threads launched: {total_blocks * block_size**2} (covers {output_size**2} output elements)')


ceil(200/16) = 13
Total blocks: 13×13 = 169
Threads launched: 43264 (covers 40000 output elements)


## Q14 — nsys profiler: which part dominates? (HtoD vs kernel vs DtoH)
Answer: **B) Host to device transfers** (26.8ms > 13.8ms > 0.16ms)


In [12]:
times = {
    'HtoD transfers':  26.8,   # ms
    'GPU kernel':      13.8,   # ms
    'DtoH transfers':   0.16,  # ms
}
total = sum(times.values())
for name, t in times.items():
    print(f'{name:20s}: {t:5.2f}ms ({100*t/total:.1f}%)')
bottleneck = max(times, key=times.get)
print(f'\nBottleneck: {bottleneck}')  # HtoD transfers


HtoD transfers      : 26.80ms (65.8%)
GPU kernel          : 13.80ms (33.9%)
DtoH transfers      :  0.16ms (0.4%)

Bottleneck: HtoD transfers


## Q18 — Max chunk size: 3 int64 columns, 24 MB RAM
Answer: **A) ~800,000 rows** (or exactly 1,000,000 with 1MB=10^6)


In [13]:
n_cols = 3
bytes_per_col = 8   # int64
bytes_per_row = n_cols * bytes_per_col
available_MB  = 24

# With 1 MB = 10^6 bytes (SI)
max_rows_SI = (available_MB * 10**6) // bytes_per_row
print(f'Bytes per row:      {bytes_per_row}')
print(f'Max rows (SI):      {max_rows_SI:,}')   # 1,000,000

# With 1 MB = 1024^2 bytes (binary)
max_rows_binary = (available_MB * 1024**2) // bytes_per_row
print(f'Max rows (binary):  {max_rows_binary:,}')  # 1,048,576
print(f'Answer A (800000) is conservative — both interpretations exceed it')


Bytes per row:      24
Max rows (SI):      1,000,000
Max rows (binary):  1,048,576
Answer A (800000) is conservative — both interpretations exceed it


## Q19 — np.memmap RAM footprint: x[::100_000] on 10^10 uint8 array
Answer: **D) ~100 KB**


In [14]:
total_elements = 10**10
step = 100_000
dtype_bytes = 1  # uint8

# x[::100_000] selects every 100_000th element
selected_elements = total_elements // step
ram_bytes = selected_elements * dtype_bytes
ram_KB = ram_bytes / 1024

print(f'Total elements:    {total_elements:,}')
print(f'Selected (step={step}): {selected_elements:,}')
print(f'RAM for y:         {ram_bytes:,} bytes = {ram_KB:.1f} KB ≈ 100 KB')
print(f'(memmap itself uses ~0 RAM — only pages you access are loaded)')


Total elements:    10,000,000,000
Selected (step=100000): 100,000
RAM for y:         100,000 bytes = 97.7 KB ≈ 100 KB
(memmap itself uses ~0 RAM — only pages you access are loaded)


## Q21 — N-body loop order: which statement is NOT correct?
Answer: **D) Loop order can be switched with no impact** (it CAN impact cache performance)


In [15]:
# Show that loop order matters for cache performance
# forces array shape (N, N) — row-major, outer i, inner j is cache-friendly

from time import perf_counter
import numpy as np

N = 500
forces = np.random.rand(N, N)

# ij order — outer i, inner j: row-major, forces[i,j] → sequential ✓
t = perf_counter()
total_ij = 0.0
for i in range(N):
    for j in range(N):
        total_ij += forces[i, j]
t_ij = perf_counter() - t

# ji order — outer j, inner i: column access, forces[i,j] → strided ✗
t = perf_counter()
total_ji = 0.0
for j in range(N):
    for i in range(N):
        total_ji += forces[i, j]
t_ji = perf_counter() - t

print(f'ij order (cache-friendly): {t_ij:.3f}s')
print(f'ji order (strided access): {t_ji:.3f}s')
print(f'Ratio: {t_ji/t_ij:.1f}x — loop order DOES matter (statement D is wrong)')


ij order (cache-friendly): 0.027s
ji order (strided access): 0.028s
Ratio: 1.0x — loop order DOES matter (statement D is wrong)


## Q24 — Can program (32s total, 16s parallelisable) reach 8s?
Answer: **B) Possibly, but further analysis needed** — we don't know if the other 16s is parallelisable


In [16]:
T_total = 32    # seconds
T_known_parallel = 16  # seconds — known parallelisable
T_unknown = T_total - T_known_parallel   # 16s unknown

# Best case: ALL 32s parallelisable → infinite speedup possible
# Worst case: only 16s parallelisable
F_min = T_known_parallel / T_total   # 0.5
S_max_min = 1 / (1 - F_min)         # max speedup if only known part is parallel

print(f'Known parallel fraction (at minimum): F = {F_min}')
print(f'S_max if only 16s parallel: {S_max_min:.1f}x')  # 2.0x
print(f'T_min achievable (if only 16s parallel): {T_total / S_max_min:.0f}s')  # 16s
print(f'\nCan we reach 8s?')
print(f'  - If other 16s is also parallel: maybe yes (T_serial→0)')
print(f'  - If other 16s is serial:        no (min time = 16s)')
print(f'  - Without knowing: cannot determine → further analysis needed')


Known parallel fraction (at minimum): F = 0.5
S_max if only 16s parallel: 2.0x
T_min achievable (if only 16s parallel): 16s

Can we reach 8s?
  - If other 16s is also parallel: maybe yes (T_serial→0)
  - If other 16s is serial:        no (min time = 16s)
  - Without knowing: cannot determine → further analysis needed
